# 05 — Weighted Risk Baseline

Purpose: create a simple transparent rule-based/scoring baseline before judging the ML models.
The baseline score is **not** a calibrated probability.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
OUTPUT_DIR = PROJECT_DIR / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "labeled_dataset.csv")
splits = pd.read_csv(DATA_DIR / "splits.csv")
df["split"] = splits["split"].values

y = df["target"].astype(int)


In [2]:
# Transparent baseline: fixed weights, deliberately not optimized to the label.
def weighted_score(frame):
    usage_scale = max(frame["usage_count"].quantile(0.95), 1.0)
    low_usage = 1 - np.clip(np.log1p(frame["usage_count"].clip(lower=0)) / np.log1p(usage_scale), 0, 1)
    inactivity = np.clip(frame["days_since_last_use"].fillna(0) / 90.0, 0, 1)
    risk = np.clip(frame["risk_weight"].fillna(0) / 10.0, 0, 1)
    wildcard = frame["is_wildcard_resource"].fillna(0).astype(float)
    return np.clip(0.45 * low_usage + 0.25 * inactivity + 0.25 * risk + 0.05 * wildcard, 0, 1)

train = df[df["split"] == "train"]
val = df[df["split"] == "validation"]
test = df[df["split"] == "test"]

# Fit only the usage scale on training data.
usage_scale = max(train["usage_count"].quantile(0.95), 1.0)

def baseline_score(frame):
    low_usage = 1 - np.clip(np.log1p(frame["usage_count"].clip(lower=0)) / np.log1p(usage_scale), 0, 1)
    inactivity = np.clip(frame["days_since_last_use"].fillna(0) / 90.0, 0, 1)
    risk = np.clip(frame["risk_weight"].fillna(0) / 10.0, 0, 1)
    wildcard = frame["is_wildcard_resource"].fillna(0).astype(float)
    return np.clip(0.45 * low_usage + 0.25 * inactivity + 0.25 * risk + 0.05 * wildcard, 0, 1)

val_score = baseline_score(val)
test_score = baseline_score(test)

print("Validation PR-AUC:", round(average_precision_score(val["target"], val_score), 4))


Validation PR-AUC: 0.4636


In [3]:
def tune_threshold(y_true, score, minimum_recall=0.90):
    rows = []
    for threshold in np.round(np.linspace(0.05, 0.95, 181), 4):
        pred = (score >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })
    table = pd.DataFrame(rows)
    eligible = table[table["recall"] >= minimum_recall]
    chosen = (eligible if len(eligible) else table).sort_values(["f1", "precision"], ascending=False).iloc[0]
    return float(chosen["threshold"]), table

baseline_threshold, threshold_table = tune_threshold(val["target"].values, val_score.values, minimum_recall=0.90)
print("Validation-selected baseline threshold:", baseline_threshold)

test_pred = (test_score >= baseline_threshold).astype(int)
print("Test precision:", round(precision_score(test["target"], test_pred, zero_division=0), 4))
print("Test recall:", round(recall_score(test["target"], test_pred, zero_division=0), 4))
print("Test F1:", round(f1_score(test["target"], test_pred, zero_division=0), 4))

Validation-selected baseline threshold: 0.435
Test precision: 0.2906
Test recall: 0.8862
Test F1: 0.4377


In [4]:
joblib_payload = {
    "model_name": "weighted_baseline",
    "weights": {"low_usage": 0.45, "inactivity": 0.25, "risk": 0.25, "wildcard": 0.05},
    "usage_scale": float(usage_scale),
    "validation_threshold": float(baseline_threshold)
}

import joblib
joblib.dump(joblib_payload, ARTIFACT_DIR / "weighted_baseline.joblib")
print("Saved baseline artifact.")

Saved baseline artifact and threshold table.
